# G1 Academy Bonus - Task 10: end-effector IK control + teleoperation UI

## Introduction
This task builds a native, position-only `ik_move_ee(hand, dx, dy, dz)` step function using the academy-supplied `hand_pose_navigation` forward/inverse-kinematics solvers, then wraps it in a small Jupyter UI to intuitively teleoperate the end effector - jog buttons for +/-X/Y/Z, plus the essential FSM mode buttons, release/re-engage buttons, and open/close-hand buttons, all in one panel, mirroring `mode_control.py`'s Dash button layout but native to the notebook.

In [ ]:
import time
from unitree_sdk2py.core.channel import ChannelFactoryInitialize, ChannelPublisher, ChannelSubscriber
from unitree_sdk2py.idl.unitree_hg.msg.dds_ import LowState_

_factory_config = None
def ensure_channel_factory(domain_id, interface):
    global _factory_config
    config = (int(domain_id), str(interface))
    if _factory_config is None:
        ChannelFactoryInitialize(*config)
        _factory_config = config
    elif _factory_config != config:
        raise RuntimeError(f"ChannelFactory already initialized as {_factory_config}; restart kernel for {config}.")
    return _factory_config

ensure_channel_factory(0, "eth0")

class Latest:
    def __init__(self, topic, message_type, queue_len=10):
        self.message = None
        self.timestamp = 0.0
        self.subscriber = ChannelSubscriber(topic, message_type)
        self.subscriber.Init(self._callback, queue_len)
    def _callback(self, message):
        self.message = message
        self.timestamp = time.time()
    def fresh(self, max_age_s=0.5):
        return self.message is not None and time.time() - self.timestamp <= max_age_s

lowstate_sub = Latest("rt/lowstate", LowState_)

## Task 1 - Reuse Task 8's upper-body pose/`arm_sdk` helpers, and Dex3 basics

In [ ]:
from unitree_sdk2py.idl.default import unitree_hg_msg_dds__LowCmd_, unitree_hg_msg_dds__HandCmd_
from unitree_sdk2py.idl.unitree_hg.msg.dds_ import LowCmd_, HandCmd_
from unitree_sdk2py.utils.crc import CRC

WAIST_JOINTS = (12, 13, 14)
UPPER_BODY_JOINTS = list(WAIST_JOINTS) + list(range(15, 22)) + list(range(22, 29))
LEFT_ARM_JOINTS = list(range(15, 22)); RIGHT_ARM_JOINTS = list(range(22, 29))
_crc = CRC()
arm_sdk_pub = ChannelPublisher("rt/arm_sdk", LowCmd_); arm_sdk_pub.Init()

def current_upper_body_pose(timeout_s=3.0):
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        if lowstate_sub.message is not None:
            return {j: float(lowstate_sub.message.motor_state[j].q) for j in UPPER_BODY_JOINTS}
        time.sleep(0.02)
    raise TimeoutError("No fresh rt/lowstate.")

def write_arm_sdk_pose(targets, weight=1.0, kp=30.0, kd=1.5, waist_kp=480.0, waist_kd=12.0):
    msg = unitree_hg_msg_dds__LowCmd_(); msg.mode_pr = 0; msg.mode_machine = 0
    msg.motor_cmd[29].q = max(0.0, min(1.0, float(weight)))
    for joint, q in targets.items():
        cmd = msg.motor_cmd[int(joint)]
        cmd.mode = 1; cmd.q = float(q); cmd.dq = 0.0; cmd.tau = 0.0
        cmd.kp = waist_kp if int(joint) in WAIST_JOINTS else kp
        cmd.kd = waist_kd if int(joint) in WAIST_JOINTS else kd
    msg.crc = _crc.Crc(msg)
    arm_sdk_pub.Write(msg)

def release_arms(steps=150, rate_hz=50.0):
    pose = current_upper_body_pose()
    for i in range(steps + 1):
        ratio = i / steps; fade = ratio * ratio * (3 - 2 * ratio); weight = 1.0 - fade
        write_arm_sdk_pose(pose, weight=weight, kp=30.0 * weight, kd=1.5 * weight, waist_kp=480.0 * weight, waist_kd=12.0 * weight)
        time.sleep(1.0 / rate_hz)

def engage_arms(steps=50, rate_hz=50.0):
    pose = current_upper_body_pose()
    for i in range(steps + 1):
        write_arm_sdk_pose(pose, weight=i / steps)
        time.sleep(1.0 / rate_hz)

HAND_CMD_TOPICS = {"left": "rt/dex3/left/cmd", "right": "rt/dex3/right/cmd"}
hand_pubs = {side: ChannelPublisher(topic, HandCmd_) for side, topic in HAND_CMD_TOPICS.items()}
for pub in hand_pubs.values():
    pub.Init()
from util import HAND_OPEN, HAND_CLOSED

def write_hand(targets, side="right", kp=0.8, kd=0.05, tau=0.02):
    msg = unitree_hg_msg_dds__HandCmd_()
    for i, q in enumerate(targets):
        cmd = msg.motor_cmd[i]
        cmd.mode = (i & 0x0F) | (1 << 4); cmd.q = float(q); cmd.dq = 0.0; cmd.tau = tau; cmd.kp = kp; cmd.kd = kd
    hand_pubs[side].Write(msg)

def open_hand(side="right"):
    return write_hand(HAND_OPEN[side], side=side)

def close_hand(side="right"):
    return write_hand(HAND_CLOSED[side], side=side)

## Task 2 - `ik_move_ee(hand, dx, dy, dz)`: position-only DLS IK step
Uses the academy-supplied `hand_pose_navigation.arm_fk.ArmFK`/`arm_ik.ArmIK` (same solvers `sdk_wrapper.ik_move_ee` uses). Solve for a small Cartesian offset from the current end-effector pose, clip the resulting joint delta to `max_dq`, then ramp to it in speed-limited, eased steps - never jump straight to the solved pose.

In [ ]:
import numpy as np
from hand_pose_navigation.arm_fk import ArmFK
from hand_pose_navigation.arm_ik import ArmIK

_fk_cache = {}
_ik_cache = {}
def fk_solver(side):
    if side not in _fk_cache:
        _fk_cache[side] = ArmFK(side, "urdf")
    return _fk_cache[side]
def ik_solver(side):
    if side not in _ik_cache:
        _ik_cache[side] = ArmIK(side, "dls", max_iter=24, tol_pos_m=0.005, tol_rot_rad=0.02)
    return _ik_cache[side]

def ik_move_ee(hand, dx=0.0, dy=0.0, dz=0.0, max_speed=0.25, max_dq=0.15, rate_hz=50.0):
    side = "right" if str(hand).lower().startswith("r") else "left"
    joints = RIGHT_ARM_JOINTS if side == "right" else LEFT_ARM_JOINTS
    current = current_upper_body_pose()
    q_init = np.array([current[j] for j in joints])
    fk = fk_solver(side)
    target_T = fk.compute_arm(q_init).copy()
    target_T[0, 3] += dx
    target_T[1, 3] += dy if side == "left" else -dy
    target_T[2, 3] += dz
    q_sol, info = ik_solver(side).solve(target_T, q_init=q_init)
    if q_sol is None:
        return {"success": False, "ik": info}
    delta = np.clip(np.asarray(q_sol) - q_init, -max_dq, max_dq)
    target_q = q_init + delta
    target = dict(current)
    for i, j in enumerate(joints):
        target[j] = float(target_q[i])
    remaining = max(abs(target[j] - current[j]) for j in joints)
    steps = max(1, int(np.ceil(remaining / max(1e-4, max_speed / rate_hz))))
    for step in range(1, steps + 1):
        ratio = step / steps; smooth = ratio * ratio * (3 - 2 * ratio)
        frame = dict(current)
        for j in joints:
            frame[j] = current[j] + (target[j] - current[j]) * smooth
        write_arm_sdk_pose(frame)
        time.sleep(1.0 / rate_hz)
    ee = tuple(float(x) for x in fk.compute_arm(target_q)[:3, 3])
    return {"success": True, "ik": info, "ee": ee, "steps": steps}

# ik_move_ee("right", dz=0.02)

## Task 3 - FSM mode helper (for the panel below)

In [ ]:
from unitree_sdk2py.g1.loco.g1_loco_client import LocoClient

FSM_IDS = {"damp": 1, "prepare": 4, "walk": 501}
loco = LocoClient(); loco.SetTimeout(5.0); loco.Init()

## Task 4 - Teleoperation panel: jog buttons + FSM + release/re-engage + open/close hand
One `ipywidgets` panel: a hand selector, a 3x2 jog grid for +/-X/Y/Z (each button fires one small `ik_move_ee` step), the essential FSM buttons (`damp`/`prepare`/`walk`), `release_arms`/`engage_arms` for `rt/arm_sdk` ownership handoff, and `open_hand`/`close_hand` - the same set of controls `mode_control.py`'s Dash app exposes, native to the notebook.

In [ ]:
import ipywidgets as widgets

STEP_M = 0.02
hand_toggle = widgets.ToggleButtons(options=["right", "left"], description="hand")

def _jog(axis, sign):
    kwargs = {"dx": 0.0, "dy": 0.0, "dz": 0.0}
    kwargs["d" + axis] = sign * STEP_M
    return ik_move_ee(hand_toggle.value, **kwargs)

btn_xm = widgets.Button(description="-X"); btn_xp = widgets.Button(description="+X")
btn_ym = widgets.Button(description="-Y"); btn_yp = widgets.Button(description="+Y")
btn_zm = widgets.Button(description="-Z"); btn_zp = widgets.Button(description="+Z")
btn_xm.on_click(lambda _b: _jog("x", -1)); btn_xp.on_click(lambda _b: _jog("x", 1))
btn_ym.on_click(lambda _b: _jog("y", -1)); btn_yp.on_click(lambda _b: _jog("y", 1))
btn_zm.on_click(lambda _b: _jog("z", -1)); btn_zp.on_click(lambda _b: _jog("z", 1))
jog_pad = widgets.GridBox(
    children=[btn_xm, btn_xp, btn_ym, btn_yp, btn_zm, btn_zp],
    layout=widgets.Layout(grid_template_columns="repeat(3, 90px)"),
)

btn_damp = widgets.Button(description="Damp", button_style="danger")
btn_prepare = widgets.Button(description="Prepare")
btn_walk = widgets.Button(description="Walk")
btn_damp.on_click(lambda _b: loco.SetFsmId(FSM_IDS["damp"]))
btn_prepare.on_click(lambda _b: loco.SetFsmId(FSM_IDS["prepare"]))
btn_walk.on_click(lambda _b: loco.SetFsmId(FSM_IDS["walk"]))
mode_row = widgets.HBox([btn_damp, btn_prepare, btn_walk])

btn_release = widgets.Button(description="Release arm_sdk")
btn_engage = widgets.Button(description="Re-engage arm_sdk")
btn_release.on_click(lambda _b: release_arms())
btn_engage.on_click(lambda _b: engage_arms())
handoff_row = widgets.HBox([btn_release, btn_engage])

btn_open = widgets.Button(description="Open hand")
btn_close = widgets.Button(description="Close hand")
btn_open.on_click(lambda _b: open_hand(hand_toggle.value))
btn_close.on_click(lambda _b: close_hand(hand_toggle.value))
hand_row = widgets.HBox([btn_open, btn_close])

ui = widgets.VBox([hand_toggle, jog_pad, mode_row, handoff_row, hand_row])
display(ui)

### Safety
Run no command cell until the subscriber state is fresh, controller ownership is known, the space is clear, and a damp path is available. Code is not invoked automatically.